# Count-Min Sketch — Numerical Verification

This notebook verifies the two worked examples used in the LinkedIn article *"How to Count Things You Can't Afford to Remember"*.

- **Example 1** (collision-free case): stream `ABACABDA` over a 3×5 grid with linear hash functions mod 5. All queries return exact counts.
- **Example 2** (collision case): stream `ABACABDAEB` over a 3×3 grid with one quadratic hash function. Demonstrates how the `min` operation rescues estimates when some rows are polluted by collisions.

Author: Antonio Leites  
Companion to LinkedIn article on Count-Min Sketch.

## Example 1 — Clean case (no collisions)

Grid: 3 rows × 5 columns. Stream: A, B, A, C, A, B, D, A. Real counts: A=4, B=2, C=1, D=1.

Hash functions:
- h₁(x) = x mod 5
- h₂(x) = (2x + 1) mod 5
- h₃(x) = (3x + 2) mod 5

In [1]:
def h1(x): return x % 5
def h2(x): return (2*x + 1) % 5
def h3(x): return (3*x + 2) % 5

letters = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
stream = ['A', 'B', 'A', 'C', 'A', 'B', 'D', 'A']
hashes = [h1, h2, h3]

# Show fingerprints
print('Fingerprints (column index per row):')
for letter, val in letters.items():
    fp = tuple(h(val) for h in hashes)
    print(f'  {letter} (val={val}): {fp}')

Fingerprints (column index per row):
  A (val=0): (0, 1, 2)
  B (val=1): (1, 3, 0)
  C (val=2): (2, 0, 3)
  D (val=3): (3, 2, 1)


In [2]:
# Build the sketch
d, w = 3, 5
M = [[0] * w for _ in range(d)]

for item in stream:
    val = letters[item]
    for row, h in enumerate(hashes):
        M[row][h(val)] += 1

print('Final matrix:')
print(f'        col 0  col 1  col 2  col 3  col 4')
for row_idx, row in enumerate(M):
    print(f'  h{row_idx+1}  ' + '  '.join(f'{v:5}' for v in row))

Final matrix:
        col 0  col 1  col 2  col 3  col 4
  h1      4      2      1      1      0
  h2      1      4      1      2      0
  h3      2      1      4      1      0


In [3]:
# Query each letter
real_counts = {letter: stream.count(letter) for letter in letters}

print('Query results:')
for letter, val in letters.items():
    cells = [M[row][h(val)] for row, h in enumerate(hashes)]
    estimate = min(cells)
    real = real_counts[letter]
    status = 'OK' if estimate == real else 'MISMATCH'
    print(f'  {letter}: cells_read={cells} -> min={estimate} (real={real}) [{status}]')

Query results:
  A: cells_read=[4, 4, 4] -> min=4 (real=4) [OK]
  B: cells_read=[2, 2, 2] -> min=2 (real=2) [OK]
  C: cells_read=[1, 1, 1] -> min=1 (real=1) [OK]
  D: cells_read=[1, 1, 1] -> min=1 (real=1) [OK]


**Observation.** With only 4 distinct letters and a 3×5 grid, every letter lands in its own clean cells in every row. No collisions, exact counts for all queries. This is pedagogically useful to show the mechanics, but does not demonstrate the collision-handling magic of CMS.

For that, we need a second example.

## Example 2 — Forced collisions (the magic of `min`)

Grid: 3 rows × 3 columns (smaller grid, more items → collisions unavoidable).  
Stream: A, B, A, C, A, B, D, A, E, B. Real counts: A=4, B=3, C=1, D=1, E=1.

Hash functions (note that h₃ is quadratic to break linearity):
- h₁(x) = x mod 3
- h₂(x) = (2x + 1) mod 3
- h₃(x) = (x² + 1) mod 3

In [4]:
def g1(x): return x % 3
def g2(x): return (2*x + 1) % 3
def g3(x): return (x*x + 1) % 3

letters5 = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
stream2 = ['A', 'B', 'A', 'C', 'A', 'B', 'D', 'A', 'E', 'B']
hashes2 = [g1, g2, g3]

# Show fingerprints
fps = {}
print('Fingerprints (column index per row):')
for letter, val in letters5.items():
    fp = tuple(h(val) for h in hashes2)
    fps[letter] = fp
    print(f'  {letter} (val={val}): {fp}')

Fingerprints (column index per row):
  A (val=0): (0, 1, 1)
  B (val=1): (1, 0, 2)
  C (val=2): (2, 2, 2)
  D (val=3): (0, 1, 1)
  E (val=4): (1, 0, 2)


In [5]:
# Detect collisions between pairs of letters
print('Collision analysis (which letter pairs land in the same cells):')
collisions_found = False
for l1 in letters5:
    for l2 in letters5:
        if l1 >= l2:
            continue
        shared_rows = [i+1 for i in range(3) if fps[l1][i] == fps[l2][i]]
        if shared_rows:
            kind = 'TOTAL collision' if len(shared_rows) == 3 else f'partial collision'
            print(f'  {l1}-{l2}: rows {shared_rows} ({kind})')
            collisions_found = True
if not collisions_found:
    print('  (none)')

Collision analysis (which letter pairs land in the same cells):
  A-D: rows [1, 2, 3] (TOTAL collision)
  B-C: rows [3] (partial collision)
  B-E: rows [1, 2, 3] (TOTAL collision)
  C-E: rows [3] (partial collision)


**Key observation.** A-D and B-E collide in all three rows (they are congruent modulo 3 and the quadratic h₃ does not save us because `0² ≡ 3²` and `1² ≡ 4²` mod 3). But **C has only a partial collision** with B and E in row 3 — and in rows 1 and 2 it sits alone. This is the pedagogically interesting case.

In [6]:
# Build the sketch
d2, w2 = 3, 3
M2 = [[0] * w2 for _ in range(d2)]

for item in stream2:
    val = letters5[item]
    for row, h in enumerate(hashes2):
        M2[row][h(val)] += 1

print('Final matrix:')
print(f'        col 0  col 1  col 2')
for row_idx, row in enumerate(M2):
    print(f'  h{row_idx+1}  ' + '  '.join(f'{v:5}' for v in row))

Final matrix:
        col 0  col 1  col 2
  h1      5      4      1
  h2      4      5      1
  h3      0      5      5


In [6]:
# Query each letter and compare to real count
real_counts2 = {letter: stream2.count(letter) for letter in letters5}

print('Query results (notice where min rescues the estimate and where it fails):')
print()
for letter, val in letters5.items():
    cells = [M2[row][h(val)] for row, h in enumerate(hashes2)]
    estimate = min(cells)
    real = real_counts2[letter]
    error = estimate - real
    verdict = 'exact' if error == 0 else f'overestimate by {error}'
    print(f'  {letter} (fingerprints={fps[letter]}):')
    print(f'     cells_read={cells}  ->  min={estimate}, real={real}  [{verdict}]')
    print()

Query results (notice where min rescues the estimate and where it fails):



NameError: name 'M2' is not defined

## Interpretation

**C is the success story.** Its cells read `[1, 1, 5]`. Row 3 is polluted by B and E (they also land in column 2 of row 3). But rows 1 and 2 are clean. The `min` operation correctly picks `1`, matching the true count.

This is the magic of CMS in action:
- Counters never go down → every cell is either correct or too high, never too low.
- With multiple independent hash rows, the chance that **all** rows are polluted for a given item is small.
- `min` picks the cleanest estimate.

**A-D and B-E are the failure mode.** They collide in every single row, so `min` has no clean cell to fall back on. This illustrates why CMS in practice uses carefully chosen **independent** hash families (typically `pairwise-independent` or better). Our toy example uses two linear hashes and one quadratic, which is not independent enough for values that are congruent mod 3.

## Formal guarantees

The grid dimensions in a real CMS are chosen based on two parameters:

- ε (epsilon) — the maximum relative error you are willing to accept
- δ (delta) — the probability that the error exceeds ε

Then:

- Width (columns): w ≈ 2/ε
- Depth (rows): d ≈ ln(1/δ)

Guarantee: the estimate is at most `real_count + ε × N` (where N is total events seen) with probability at least `1 − δ`.

For example, to get error below 0.1% of total events with failure probability below one in a million, you need roughly 2,000 columns × 14 rows ≈ 28,000 counters. A few hundred kilobytes. For a stream of billions of events. That is the trade-off CMS offers.

## Further reading

- Cormode, G., & Muthukrishnan, S. (2005). *An improved data stream summary: the Count-Min sketch and its applications.* Journal of Algorithms, 55(1), 58–75.
- Zhao, H. et al. (2026). *Exponential quantum advantage in processing massive classical data.* arXiv:2604.07639 — extends the sketching philosophy to quantum computation via "quantum oracle sketching".